# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant-python/) library.

### Dataset Source
This dataset is described via a machine-readable [Croissant schema](https://mlcommons.org/croissant/) accessible at:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if needed
!pip install -q mlcroissant

## 1. Data Loading
In this section, we load the Croissant dataset schema and create a `mlcroissant.Dataset` object. We preview its metadata to understand the dataset at a high level.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Create mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Print basic metadata (using attribute access to avoid subscripting)
print('Title:', dataset.metadata.name)
print('Identifier:', dataset.metadata.identifier)
print('Version:', dataset.metadata.version)
print('Description:', dataset.metadata.description)
print('Authors:')
for a in getattr(dataset.metadata, 'author', []):
    print(' -', getattr(a, 'name', getattr(a, '@id', str(a))))
print('\nKeywords:', getattr(dataset.metadata, 'keywords', []))

## 2. Data Overview
Let's inspect which **Record Sets** are present in the dataset schema and list their `@id`, fields, and column information. This helps us understand the structural building blocks to query data.

In [ ]:
# List all record sets, their @id, and associated fields
record_sets = list(dataset.get_record_sets())

print(f'Total record sets: {len(record_sets)}\n')
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print('  Fields:')
    # The field can be an @id reference or embedded dict
    for f in fields:
        if isinstance(f, dict):
            print(f"    - {f.get('@id', f)}  (label: {f.get('name', f.get('rdfs:label', '-') )})")
        else:
            print(f"    - {f}")
    columns = rs.get('column', [])
    if columns:
        if not isinstance(columns, list):
            columns = [columns]
        print('  Columns:')
        for c in columns:
            if isinstance(c, dict):
                print(f"    - {c.get('@id', c)}  (label: {c.get('name', c.get('rdfs:label', '-') )})")
            else:
                print(f"    - {c}")
    print('')

## 3. Data Extraction
We will extract data from a chosen record set. **All references use their `@id`.**

Below, the Record Sets are loaded into `DataFrame` objects. Use the output above to select record set and field `@id`s as variables.

In [ ]:
# Prepare to extract all data
dataframes = {}

# List of all record set @id's (from overview, to automate)
record_set_ids = [rs['@id'] for rs in dataset.get_record_sets()]

for record_set_id in record_set_ids:
    try:
        print(f"Loading records for {record_set_id} ...")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            # Normalize field keys to strings (if using dicts)
            df.columns = [str(col) for col in df.columns]
            dataframes[record_set_id] = df
            print(f" - Loaded {len(df)} records with columns: {df.columns.tolist()}\n")
        else:
            print(' - No records found.')
    except Exception as e:
        print(f' - Skipped due to error: {e}')
if not dataframes:
    print('No tabular record sets found. Please check schema definition.')

### Example: Preview Data
Let's pick one record set (by `@id`) and display the first few rows and available columns.

In [ ]:
# Choose a record set to explore in detail. Replace the value with any valid @id from above.
if dataframes:
    sample_record_set_id = list(dataframes.keys())[0]  # pick the first as default
    print(f"Columns in '{sample_record_set_id}':")
    print(dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's analyze the main data table. We'll select a numeric field and perform basic filtering and normalization, always referencing fields by their `@id`. 

**Modify the variable assignments below to match your schema's actual field `@id`s if needed.**

In [ ]:
import numpy as np
import warnings

if dataframes:
    record_set_id = sample_record_set_id  # the record set you're working with
    df = dataframes[record_set_id]
    print('First 5 columns:', df.columns[:5].tolist())
    # Heuristic: look for a likely numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[col])] 
    if not numeric_candidates:
        # Try to coerce numeric columns (if all are objects/strings)
        for col in df.columns:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                converted = pd.to_numeric(df[col], errors='coerce')
                if converted.notna().sum() > 2:  # at least two numeric values
                    numeric_candidates.append(col)
        print(f'Numeric candidate fields by conversion: {numeric_candidates}')
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # pick first
        print(f"Using numeric field: '{numeric_field_id}'")
        # Convert to float
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Example: filter for values > median
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}")
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group by a likely categorical field
        group_fields = [col for col in df.columns if ('group' in col.lower() or 'type' in col.lower() or 'sample' in col.lower()) and col!=numeric_field_id]
        if group_fields:
            group_field = group_fields[0]
            print(f'Grouping by: {group_field}')
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print('No obvious group field found.')
    else:
        print('No numeric fields found for EDA.')

## 5. Visualization
Let's visualize the distribution of a selected numeric field and inspect the effect of filtering and normalization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if dataframes and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if 'filtered_df' in locals():
        plt.figure(figsize=(10,4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True, color='orange')
        plt.title(f"Normalized {numeric_field_id} (filtered)")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel('Count')
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² Croissant-formatted dataset describing mass spectrometry analysis of proteins from extracellular vesicles of stimulated human mast cells. We programmatically loaded schema and tabular data, inspected its structure by `@id`, extracted records into DataFrames, filtered and normalized a representative numeric field, and visualized their distribution.

**Key Findings**:
- The scheme provides a rich description of experiment-derived protein data with explicit field and record set references.
- Data extraction and EDA workflows are naturally powered by referencing entities by `@id`, supporting both machine-driven and human-readable context for provenance and interoperability.

Further analysis could focus on domain-specific interpretation, such as biological pathways, comparisons across experimental conditions, or integrating with UniProt annotations. Checkout [mlcroissant documentation](https://mlcommons.github.io/croissant-python/) for advanced usage!